In [127]:
import pandas as pd
import os
import ast


class Pattern:
    country_a_id    = ""
    country_b_id    = ""
    pattern_length  = ""
    start_year_a_fk = ""
    start_year_b_fk = ""
    indexes         = ""
    avg_corr        = ""
    orgs            = ""
    power           = ""
    sectors         = []

class PatternISCICouple:
    pi_tuple = ()

class CountryAPatterns:
    """ caps stands for: country a patterns """
    caps = []

countries_df = pd.read_csv(r"/workspaces/qgi-4.0/data/countries.csv")
patterns = pd.read_csv(r"/workspaces/qgi-4.0/data/ppr.csv")
ISCIS_CSVS = r"/workspaces/qgi-4.0/data/countries_isci"


def get_country_a_icsis(country_a_id, folder_path = ISCIS_CSVS):
    country_a_id = int(country_a_id)
    if country_a_id > 0 and country_a_id < 251:
        country_a_id = str(country_a_id)
        for filename in os.listdir(folder_path):
            if filename.endswith('_solos_patterns.csv'):
                id_part = filename.split('_')[0]
                if id_part == country_a_id:
                    file_path = os.path.join(folder_path, filename)
                    try:
                        return pd.read_csv(file_path)
                    except Exception as e:
                        print(f"Error reading {filename}: {e}")
    print("No pattern file found for the specified country ID.")
    return None

def get_country_a_patterns(country_a_id, folder_path = ISCIS_CSVS):
    country_a_id = int(country_a_id)
    if country_a_id > 0 and country_a_id < 251:
        country_a_id = str(country_a_id)
        for filename in os.listdir(folder_path):
            if "_solos_" not in filename and filename.endswith('_patterns.csv'):
                id_part = filename.split('_')[0]
                if id_part == country_a_id:
                    file_path = os.path.join(folder_path, filename)
                    try:
                        return pd.read_csv(file_path)
                    except Exception as e:
                        print(f"Error reading {filename}: {e}")
    print("No pattern file found for the specified country ID.")
    return None

def make_pattern(row):
    pattern = Pattern()
    pattern.country_a_id    = int(row["country_a_id_fk"])
    pattern.country_a       = get_name_by_id(pattern.country_a_id)
    pattern.country_b_id    = int(row["country_b_id_fk"])
    pattern.country_b       = get_name_by_id(pattern.country_b_id)
    pattern.pattern_length  = int(row["pattern_length_fk"])
    pattern.start_year_a_fk = int(row["start_year_a_fk"])
    pattern.start_year_b_fk = int(row["start_year_b_fk"])
    pattern.indexes         = int(row["indexes"])
    pattern.avg_corr        = round(row["avg_corr"], 6)
    pattern.orgs            = int(row["orgs"])
    pattern.power           = round(row["Power"], 6)
    pattern.sectors         = ast.literal_eval(row["Sectors"])
    return pattern

def show_pattern(pattern):
    print("country_a: "       + str(pattern.country_a))
    print("country_a_id: "    + str(pattern.country_a_id))
    print("country_b: "       + str(pattern.country_b))
    print("country_b_id: "    + str(pattern.country_b_id))
    print("pattern_length: "  + str(pattern.pattern_length))
    print("start_year_a_fk: " + str(pattern.start_year_a_fk))
    print("start_year_b_fk: " + str(pattern.start_year_b_fk))
    print("indexes: "         + str(pattern.indexes))
    print("avg_corr: "        + str(pattern.avg_corr))
    print("orgs: "            + str(pattern.orgs))
    print("power: "           + str(pattern.power))
    print("sectors: "         + str(pattern.sectors))

def get_icsi_for_pattern(pattern):
    icsis = get_country_a_icsis(pattern.country_a_id)
    icsis = icsis[icsis["country_a_id_fk"]   == pattern.country_a_id]
    icsis = icsis[icsis["country_b_id_fk"]   == pattern.country_b_id]
    icsis = icsis[icsis["pattern_length_fk"] == pattern.pattern_length]
    icsis = icsis[icsis["start_year_a_fk"]   == pattern.start_year_a_fk]
    icsis = icsis[icsis["start_year_b_fk"]   == pattern.start_year_b_fk]
    return icsis

def get_id_by_name(name):
    """
    Searches all name columns (name_1 to name_20) for the input name and returns the corresponding country ID.
    """
    for _, row in countries_df.iterrows():
        for i in range(1, 21):
            column = f"name_{i}"
            if column in row:
                current_value = row[column]
                if pd.notna(current_value) and current_value == name:
                    return row["id"]
    return None

def get_name_by_id(input_id):
    """
    Searches for the specified country ID and returns the primary name (name_1).
    """
    try:
        id_int = int(input_id)
        result = countries_df[countries_df["id"] == id_int]
        if not result.empty:
            return result["name_1"].iloc[0]
    except ValueError:
        pass  # Invalid ID format
    return None

def check_half_similarity(list1, list2):
    """
    Returns True if either list contains at least half of the elements of the other list.
    """
    # Create sets for efficient lookups
    set_list2 = set(list2)
    count1 = sum(1 for element in list1 if element in set_list2)
    
    set_list1 = set(list1)
    count2 = sum(1 for element in list2 if element in set_list1)
    
    # Check if either condition is met
    return (count1 >= len(list1) / 2) or (count2 >= len(list2) / 2)

    

In [124]:
#for country_a_id in range(1, 251):
for country_a_id in range(1, 2):
    country_a_patterns      = CountryAPatterns()
    country_a_patterns.caps = []
    country_a               = get_name_by_id(country_a_id)
    country_a_patterns_df   = get_country_a_patterns(country_a_id)
    for i, row in country_a_patterns_df.iterrows():
        
        country_a_patterns.caps.append(make_pattern(row))
    # 
    # isci    = get_icsi_for_pattern(pattern)
    # pattern_isci = PatternISCICouple()
    # pattern_isci.pi_tuple = (pattern, isci)
    # country_a_patterns_list.cap_list.append(pattern_isci.pi_tuple)
    break

len(country_a_patterns.caps)

1059

In [125]:
for i, pattern in enumerate(country_a_patterns.caps):
    if i + 1 < len(country_a_patterns.caps):
        pattern_1 = pattern
        pattern_2 = country_a_patterns.caps[i + 1]
        
        
        break
    else:
        continue

In [ ]:
if pattern_1.sectors == pattern_2.sectors:
    print("same secotrs")
elif check_half_similarity(pattern_1.sectors, pattern_2.sectors):
    if pattern_1.indexes 


In [128]:
show_pattern(pattern_1)

country_a: Afghanistan
country_a_id: 1
country_b: Lebanon
country_b_id: 121
pattern_length: 3
start_year_a_fk: 2015
start_year_b_fk: 2018
indexes: 2
avg_corr: 0.97663
orgs: 1
power: 0
sectors: ['Unknown']


In [129]:
show_pattern(pattern_2)

country_a: Afghanistan
country_a_id: 1
country_b: Paraguay
country_b_id: 172
pattern_length: 9
start_year_a_fk: 2009
start_year_b_fk: 2012
indexes: 2
avg_corr: 0.920635
orgs: 1
power: 0
sectors: ['Political State Sector', 'Social Sector']
